# CNN-GAF Cloudburst Prediction â Model Training

**Paper:** Precision forecasting of cloudbursts with CNNs and GAF for real-time disaster response
**Journal:** Modeling Earth Systems and Environment, 12, 184 (2026)
**DOI:** [10.1007/s40808-026-02826-4](https://doi.org/10.1007/s40808-026-02826-4)

This notebook performs hyperparameter search and trains CNN models on GAF-encoded meteorological data for 6-hour and 12-hour cloudburst prediction.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from sklearn.metrics import confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")

## Configuration

In [ ]:
# Data files
CB_DATA   = 'CB6.npy'    # Switch to 'CB12.npy' for 12-hour model
NB_DATA   = 'NB.npy'
TEST_DATA = 'TEST6.npy'  # Switch to 'TEST12.npy' for 12-hour model

# Output directory
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Training settings
EPOCHS     = 50
BATCH_SIZE = 16
VAL_SPLIT  = 0.3
IMAGE_SIZE = 256
N_CHANNELS = 6   # Number of meteorological features

# Hyperparameter grid
OPTIMIZERS    = ['adam', 'rmsprop']
KERNEL_SIZES  = [(1, 1), (2, 2)]
POOLING_SIZES = [(2, 2), (3, 3)]
LEARNING_RATES = [0.0005, 0.001]

## 1. Data Loading

In [ ]:
cloudburst_data     = np.load(CB_DATA)
non_cloudburst_data = np.load(NB_DATA)

print(f"Cloudburst samples:     {cloudburst_data.shape}")
print(f"Non-cloudburst samples: {non_cloudburst_data.shape}")

# Combine and label
X = np.concatenate((cloudburst_data, non_cloudburst_data))
y = np.concatenate((np.ones(cloudburst_data.shape[0]),
                    np.zeros(non_cloudburst_data.shape[0])))

# Shuffle (save index for reproducibility)
rng = np.random.default_rng(seed=42)
shuffle_index = rng.permutation(X.shape[0])
X, y = X[shuffle_index], y[shuffle_index]

print(f"\nCombined dataset: X={X.shape}, y={y.shape}")
print(f"Class balance â CB: {int(y.sum())}  NB: {int((1-y).sum())}")

## 2. Sample GAF Visualisation

In [ ]:
FEATURE_LABELS = [
    "Rainfall (mm)", "Temperature (Â°C)", "Relative Humidity (%)",
    "Wind Speed 10m (Kt)", "SLP (hPa)", "MSLP (hPa)"
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle("GAF Images â Cloudburst Sample", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    im = ax.imshow(X[0][i], cmap='viridis', aspect='auto')
    ax.set_title(FEATURE_LABELS[i], fontsize=9)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'sample_gaf.png'), dpi=150)
plt.show()

## 3. Model Architecture

In [ ]:
def build_model(kernel_size, pooling_size, optimizer, learning_rate, name='cnn_gaf'):
    """
    Build a CNN model for GAF image classification.
    Input shape: (N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE) â channels-first
    """
    model = Sequential(name=name)
    model.add(InputLayer(input_shape=(N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)))

    model.add(Conv2D(128, kernel_size=kernel_size, activation='relu',
                     padding='same', data_format='channels_first'))
    model.add(MaxPooling2D(pool_size=pooling_size, padding='same',
                           data_format='channels_first'))

    model.add(Conv2D(256, kernel_size=kernel_size, activation='relu',
                     padding='same', data_format='channels_first'))
    model.add(MaxPooling2D(pool_size=(1, 1), padding='same',
                           data_format='channels_first'))

    model.add(Conv2D(512, kernel_size=kernel_size, activation='relu',
                     padding='same', data_format='channels_first'))
    model.add(MaxPooling2D(pool_size=pooling_size, padding='same',
                           data_format='channels_first'))

    model.add(Flatten())
    model.add(Dense(256, activation='relu', kernel_regularizer='l1_l2'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    opt_map = {'adam': Adam, 'sgd': SGD, 'rmsprop': RMSprop}
    opt = opt_map[optimizer](learning_rate=learning_rate)
    model.compile(optimizer=opt, loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

## 4. Hyperparameter Search

In [ ]:
results = []
best_auc, best_path = 0.0, None

total = len(OPTIMIZERS) * len(KERNEL_SIZES) * len(POOLING_SIZES) * len(LEARNING_RATES)
run = 0

for opt in OPTIMIZERS:
    for k in KERNEL_SIZES:
        for p in POOLING_SIZES:
            for lr in LEARNING_RATES:
                run += 1
                tag = f"{opt}_k{k[0]}_p{p[0]}_lr{str(lr).replace('.','')}"
                print(f"[{run}/{total}] {tag}")

                K.clear_session()

                try:
                    model = build_model(k, p, opt, lr, name=tag)

                    history = model.fit(
                        X, y,
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        validation_split=VAL_SPLIT,
                        verbose=0
                    )

                    y_pred_prob = model.predict(X, verbose=0).flatten()
                    y_pred      = np.round(y_pred_prob)
                    cm          = confusion_matrix(y, y_pred)
                    auc         = roc_auc_score(y, y_pred_prob)
                    acc         = (cm[0,0] + cm[1,1]) / cm.sum()

                    save_path = os.path.join(MODELS_DIR, f'{tag}.h5')
                    model.save(save_path)

                    results.append({
                        'tag': tag, 'optimizer': opt,
                        'kernel': k, 'pooling': p, 'lr': lr,
                        'val_acc': round(history.history['val_accuracy'][-1], 4),
                        'train_acc': round(acc, 4),
                        'auc': round(auc, 4)
                    })

                    if auc > best_auc:
                        best_auc  = auc
                        best_path = save_path

                    print(f"  â AUC: {auc:.4f}  Val Acc: {history.history['val_accuracy'][-1]:.4f}")

                except Exception as e:
                    print(f"  â Failed: {e}")

print(f"\nBest AUC: {best_auc:.4f}  â  {best_path}")

## 5. Results Summary

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results).sort_values('auc', ascending=False)
print(df_results.to_string(index=False))

# Plot validation accuracy across runs
plt.figure(figsize=(10, 4))
plt.bar(df_results['tag'], df_results['auc'])
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.ylabel('ROC-AUC')
plt.title('Hyperparameter Search â ROC-AUC Comparison')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'hparam_comparison.png'), dpi=150)
plt.show()